# Hierarchical Clustering

> **Dependência:** Este notebook assume que o `eda.ipynb` foi corrido primeiro  
> e que os ficheiros `costumer_preprocessed_combined.csv`, `customer_info.csv`  
> e `customer_basket.csv` estão na mesma pasta.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

## 1. Load Pre-processed Data

In [ ]:
# Scaled data — used for fitting the model
costumer_preprocessed = pd.read_csv("costumer_preprocessed_combined.csv")

# Basket data
costumer_basket = pd.read_csv("customer_basket.csv")

# Original data — used for human-readable cluster profiles
costumer_raw = pd.read_csv("customer_info.csv")

# Reconstruct aligned customer_id list (same inner merge as EDA)
basket_agg = (
    costumer_basket
    .groupby('customer_id')
    .agg(total_transactions=('invoice_id', 'count'))
    .reset_index()
)
costumer = pd.merge(costumer_raw, basket_agg, on='customer_id', how='inner').reset_index(drop=True)

print(f"Preprocessed shape : {costumer_preprocessed.shape}")
print(f"Customer shape     : {costumer.shape}")

## 2. Dendrogram — Choose Number of Clusters

We use `scipy`'s `linkage` + `dendrogram` to visualise the hierarchical  
merging process.  
A horizontal cut across the dendrogram at a given height determines the  
number of clusters — **the number of vertical lines the cut crosses**.

We compare two linkage methods, as recommended in class:
- **Ward** — minimises within-cluster variance; tends to produce balanced, spherical clusters.
- **Single** (minimum) — can form elongated chains; useful to detect outliers.

In [ ]:
def plot_dendrogram(model, ax, title, color_threshold=None, **kwargs):
    """Plot a scipy dendrogram from a fitted AgglomerativeClustering model."""
    # Build linkage matrix from the fitted model
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1
        else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack(
        [model.children_, model.distances_, counts]
    ).astype(float)

    dendrogram(linkage_matrix, ax=ax, color_threshold=color_threshold, **kwargs)
    ax.set_title(title)
    ax.set_xlabel("Number of points in node (or index of point if no parenthesis)")
    ax.set_ylabel("Distance")

In [ ]:
# Fit with distance_threshold=0 and n_clusters=None to get the full tree
ward_full = AgglomerativeClustering(
    linkage='ward', distance_threshold=0, n_clusters=None
).fit(costumer_preprocessed)

single_full = AgglomerativeClustering(
    linkage='single', distance_threshold=0, n_clusters=None
).fit(costumer_preprocessed)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_dendrogram(
    ward_full, axes[0],
    title="Ward Linkage Dendrogram",
    truncate_mode='level', p=5
)

plot_dendrogram(
    single_full, axes[1],
    title="Single Linkage Dendrogram",
    truncate_mode='level', p=5
)

plt.tight_layout()
plt.show()

In [ ]:
# Ward dendrogram with horizontal cut — adjust y to match a natural gap
fig, ax = plt.subplots(figsize=(10, 6))
plot_dendrogram(
    ward_full, ax,
    title="Ward Linkage — cut at k=6",
    truncate_mode='level', p=5
)
# The y-value should be set where the longest vertical lines are visible
# Adjust based on the dendrogram output above
ax.axhline(y=50, color='red', linestyle='--', label='Cut (k=6)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Silhouette Score — Validate K

As seen in class, the elbow / dendrogram gives us candidate k values.  
We validate them with the **silhouette score**.

In [ ]:
sil_scores = {}
for k in range(2, 12):
    labels = AgglomerativeClustering(linkage='ward', n_clusters=k).fit_predict(costumer_preprocessed)
    sil_scores[k] = silhouette_score(costumer_preprocessed, labels)

best_k = max(sil_scores, key=sil_scores.get)
print(f"Best k by silhouette (Ward): {best_k}  (score={sil_scores[best_k]:.4f})")

plt.figure(figsize=(8, 4))
plt.bar(sil_scores.keys(), sil_scores.values(), color='steelblue', edgecolor='black')
plt.xlabel('k')
plt.ylabel('Silhouette score')
plt.title('Silhouette scores — Ward Hierarchical')
plt.tight_layout()
plt.show()

## 4. Fit Final Models — Ward & Single

In [ ]:
chosen_k = 6  # adjust based on dendrogram + silhouette analysis above

ward = AgglomerativeClustering(linkage='ward', n_clusters=chosen_k)
single = AgglomerativeClustering(linkage='single', n_clusters=chosen_k)

costumer_preprocessed['cluster_ward']   = ward.fit_predict(costumer_preprocessed)
costumer_preprocessed['cluster_single'] = single.fit_predict(costumer_preprocessed)

print(f"Ward   distribution:\n{pd.Series(ward.labels_).value_counts().sort_index()}\n")
print(f"Single distribution:\n{pd.Series(single.labels_).value_counts().sort_index()}")

In [ ]:
features = costumer_preprocessed.drop(columns=['cluster_ward', 'cluster_single'])

sil_ward   = silhouette_score(features, ward.labels_)
sil_single = silhouette_score(features, single.labels_)

print(f"Silhouette — Ward  (k={chosen_k}): {sil_ward:.4f}")
print(f"Silhouette — Single(k={chosen_k}): {sil_single:.4f}")

## 5. Cluster Profiles — Ward

In [ ]:
costumer['cluster_ward']   = ward.labels_
costumer['cluster_single'] = single.labels_

In [ ]:
profile_cols = costumer.drop(
    columns=['customer_id', 'customer_name', 'customer_birthdate', 'cluster_single'],
    errors='ignore'
)

print("── Ward cluster means ──")
print(profile_cols.groupby('cluster_ward').mean().T)

print("\n── Overall means ──")
print(profile_cols.drop(columns='cluster_ward').mean())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

pd.Series(ward.labels_).value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black'
)
axes[0].set_title(f'Cluster sizes — Ward (k={chosen_k})')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Number of customers')

pd.Series(single.labels_).value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='black'
)
axes[1].set_title(f'Cluster sizes — Single (k={chosen_k})')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Number of customers')

plt.tight_layout()
plt.show()

## 6. Heatmap — Cluster vs. Feature (Ward)

In [ ]:
profile_scaled = (
    costumer_preprocessed
    .drop(columns='cluster_single')
    .groupby('cluster_ward')
    .mean()
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    profile_scaled.T,
    cmap='RdBu_r', center=0,
    linewidths=0.5, ax=ax, annot=False
)
ax.set_title(f'Mean scaled feature value per cluster — Ward (k={chosen_k})')
ax.set_xlabel('Cluster')
plt.tight_layout()
plt.show()

## 7. Compare Ward vs. Single Linkage

As shown in class, single linkage tends to form chains while Ward produces  
more balanced, spherical clusters.

In [ ]:
comparison = pd.crosstab(
    costumer['cluster_ward'],
    costumer['cluster_single'],
    rownames=['Ward'],
    colnames=['Single']
)
print(comparison)

## 8. Export Cluster Assignments

In [ ]:
output = costumer[['customer_id']].copy()
output['cluster_ward']   = ward.labels_
output['cluster_single'] = single.labels_
output.to_csv("hierarchical_cluster_assignments.csv", index=False)
print(f"Saved {len(output)} rows → hierarchical_cluster_assignments.csv")
output.head()